# TR_ABDOMEN_RAD dry run

Validates the fixed pipeline on Kaggle:

1. assemble fixed labels onto prepared images
2. full dataset sanity gate (must print PASS)
3. build a small subset
4. 1-epoch GPU smoke of YOLO11-L-P2 @1280 (T4 has no bf16, so fp16)
5. patient-level prediction and evaluation with anatomical gates

This validates plumbing, not accuracy.

In [ ]:
import os
import torch

print("torch", torch.__version__)
print("cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print("input:", sorted(os.listdir("/kaggle/input")))

In [ ]:
!pip -q install ultralytics==8.4.93 scikit-learn kagglehub

In [ ]:
import os
import shutil

code_root = "/kaggle/input/tr-abdomen-rad-code"
if not os.path.isdir(code_root):
    import kagglehub
    code_root = kagglehub.dataset_download("huseyincavus/tr-abdomen-rad-code")
print("code_root:", code_root, sorted(os.listdir(code_root)))
for d in ("src", "trainer", "configs", "annotations"):
    shutil.copytree(os.path.join(code_root, d), f"/kaggle/working/{d}", dirs_exist_ok=True)

In [ ]:
import os

def pick(paths):
    for p in paths:
        if os.path.isdir(p):
            return p
    raise FileNotFoundError(paths)

IMAGES = pick([
    "/kaggle/input/abdomen-prepared/yolo_dataset/images",
    "/kaggle/input/datasets/huseyincavus/abdomen-prepared/yolo_dataset/images",
])
LABELS = pick([
    "/kaggle/input/abdomen-yolo-labels/labels",
    "/kaggle/input/datasets/huseyincavus/abdomen-yolo-labels/labels",
])
print("images:", IMAGES)
print("labels:", LABELS)

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-u", "/kaggle/working/src/assemble_dataset.py",
                "--images", IMAGES, "--labels", LABELS, "--out", "/kaggle/working/ds"], check=True)

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-u",
                "/kaggle/working/trainer/sanity_check.py",
                "--images",
                "/kaggle/working/ds/images",
                "--labels",
                "/kaggle/working/ds/labels",
                "--out",
                "/kaggle/working/sanity.json",
                "--montage",
                "/kaggle/working/montage",
                "--image-size",
                "1280"], check=True)

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-u",
                "/kaggle/working/src/make_subset.py",
                "--src",
                "/kaggle/working/ds",
                "--out",
                "/kaggle/working/ds_mini",
                "--train",
                "1200",
                "--val",
                "400",
                "--test",
                "200",
                "--per-patient",
                "2"], check=True)

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-u",
                "/kaggle/working/trainer/train.py",
                "--data",
                "/kaggle/working/ds_mini/dataset.yaml",
                "--model",
                "/kaggle/working/configs/yolo26x-p2.yaml",
                "--weights", "yolo26x.pt",
                "--epochs",
                "1",
                "--batch",
                "4",
                "--imgsz",
                "1280",
                "--amp-dtype",
                "fp16",
                "--device",
                "0",
                "--workers",
                "2",
                "--name",
                "dry-run",
                "--save-period",
                "-1",
                "--exist-ok"], check=True)

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-u",
                "/kaggle/working/src/patient_eval.py",
                "predict",
                "--model",
                "/kaggle/working/runs/detect/abdomen/dry-run/weights/best.pt",
                "--images",
                "/kaggle/working/ds_mini/images/val",
                "--out",
                "/kaggle/working/pred_val.csv",
                "--imgsz",
                "1280",
                "--conf",
                "0.005",
                "--max-det",
                "30",
                "--batch",
                "4",
                "--device",
                "0"], check=True)

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-u",
                "/kaggle/working/src/patient_eval.py",
                "eval",
                "--val-pred",
                "/kaggle/working/pred_val.csv",
                "--test-pred",
                "/kaggle/working/pred_val.csv",
                "--val-images",
                "/kaggle/working/ds_mini/images/val",
                "--test-images",
                "/kaggle/working/ds_mini/images/val",
                "--gates",
                "/kaggle/working/annotations/slice_gates.csv",
                "--score",
                "topk",
                "--topk",
                "3",
                "--grid",
                "--bootstrap",
                "200",
                "--out",
                "/kaggle/working/eval_dry"], check=True)

Done. Expected: sanity `PASS` with 0 violations, 1 epoch completed, and `eval_dry/report.json` written.